In [1]:
import numpy as np
from qutip import *
import plotly.graph_objects as go
from tqdm import tqdm,trange
import os
from datetime import datetime
timestamp = datetime.now().strftime('%M-%S')

from scipy.interpolate import interp1d
from sklearn.metrics import mean_absolute_error

In [2]:
# --- Updated signal function ---
def signal(data, samples, first='reference'):
    data_shape = data.shape[0]
    steps = int(data_shape / (2 * samples))

    if first.lower() == 'reference':
        reference_samples = np.mean(np.reshape(data[::2], (steps, samples)), axis=1)
        signal_samples = np.mean(np.reshape(data[1::2], (steps, samples)), axis=1)
    elif first.lower() == 'signal':
        signal_samples = np.mean(np.reshape(data[::2], (steps, samples)), axis=1)
        reference_samples = np.mean(np.reshape(data[1::2], (steps, samples)), axis=1)
    else:
        raise ValueError("`first` must be either 'reference' or 'signal'")

    signal_photon = signal_samples / reference_samples
    return signal_photon, reference_samples, signal_samples

# --- Wrapper function ---
def data_to_time_signal(data, samples, first='reference'):
    time = data['time_axis']
    signal_photon, reference_samples, signal_samples = signal(data['avg_data'], samples, first=first)
    return time, signal_photon, reference_samples, signal_samples

In [3]:
from plotly import graph_objs as go
# Figure format for plotly
fig_template = go.layout.Template()
fig_template.layout = {
    'template': 'simple_white+presentation',
    'autosize': False,
    'width': 1120,
    'height': 450,
    # 'opacity': 0.2,
    'xaxis': {
        # 'title': 'Time (\u03BCs)',
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 1.5,
        'tickwidth': 1.5,
        'ticklen': 5,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white',
        },
    'yaxis': {
        # 'title': 'Coherence',
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 1.5,
        'tickwidth': 1.5,
        'ticklen': 5,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white'
        },
    'font':{'family':'mathjax',
            'size': 30,
            }
}

In [4]:
def get_B1(t_pi):
    return (2*np.pi/28025)*(1/(2*t_pi))

###### References 

1. Zhiyang Yuan et. al. 'Charge state dynamics and optically detected electron spin resonance contrast of shallow nitrogen-vacancy centers in diamond", DOI: https://doi.org/10.1103/PhysRevResearch.2.033263
2. I. Panadero et. al. ""Photon-emission statistics for single nitrogen-vacancy centers, DOI: https://doi.org/10.1103/PhysRevApplied.22.014035"

In [5]:
def nv_parameters(gamma_rel=0.0001, gamma_deph=0.001, power_laser=100, B1=0, theta=np.pi/2, phi=0,Bx=0.0,By=0.0,Bz=0.0,
                  gamma_d_in=0,gamma_d_out=0,exc_rate=0.1,rel_rate=10,s0_in=12,s1_in=80,s0_out=3.3,s1_out=2.4):
    Bx = Bx                          # Static magnetic field along x in tesla
    By = By                          # Static magnetic field along y in tesla
    Bz = Bz                          # Static magnetic field along z in tesla
    c = exc_rate                      # NV- Laser excitation constant in MHz/uW 
    
    gamma_p = c*power_laser          # NV- Laser excitation rate
    # gamma_0 = 63                   # NV- Radiative emission rate 
    gamma_0 = rel_rate                     # NV- Radiative emission rate 
        
    gamma_s0_in = s0_in                 # NV- Excited 0 to singlet
    gamma_s1_in = s1_in                 # NV- Excited 1 to singlet
    gamma_s0_out = s0_out               # NV- Singlet to ground 0 
    gamma_s1_out = s1_out               # NV- Singlet to ground 1

    if power_laser!=0:
        dark_ratio = 50/power_laser
    else:
        dark_ratio = 1

    # print(exc_rate,rel_rate,s0_in,s1_in,s0_out,s1_out,gamma_d_in,gamma_d_out)
    
    # if gamma_d_in==0 or gamma_d_out==0:
    #     gamma_d_in = 0
    #     gamma_d_out = 0
        
    gamma_d0_in =  gamma_d_in
    gamma_d1_in =  gamma_d_in
    gamma_d0_out =  gamma_d_out
    gamma_d1_out = gamma_d_out

    # print(gamma_d0_in, gamma_d1_in, gamma_d0_out, gamma_d1_out)
    
    B1x = B1 * np.sin(theta) * np.cos(phi)
    B1y = B1 * np.sin(theta) * np.sin(phi)
    B1z = B1 * np.cos(theta)
    
    parameters = [Bx, By, Bz, gamma_rel, gamma_deph, gamma_p, gamma_0, gamma_s0_in, gamma_s1_in, gamma_s0_out, gamma_s1_out,
                  gamma_d0_in, gamma_d1_in, gamma_d0_out, gamma_d1_out, B1x, B1y, B1z]

    return parameters

In [6]:
def NV_spinNcharge(parameters, tlist, psi0, drive=-1,detuning_minus = 0.0,E=0.0):

    Bx, By, Bz, gamma_rel, gamma_deph, gamma_p, gamma_0, gamma_f0, gamma_f1, gamma_s0, gamma_s1, gamma_d0_in, gamma_d1_in, gamma_d0_out, gamma_d1_out, B1x, B1y, B1z = parameters

    # Hamiltonian Parameters
    D = 2870          # MHz
    g = 28025         # MHz/T
    E = E             # MHz

    # Operators
    n_states = 8
    spin_states = 3
    rho = np.zeros((n_states,n_states),dtype=complex)

    # Pauli Operators
    sx = rho.copy()
    sx[:spin_states,:spin_states] = jmat(1,'x').full()
    sx[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    sx = Qobj(sx)
    sy = rho.copy()
    sy[:spin_states,:spin_states] = jmat(1,'y').full()
    sy[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    sy = Qobj(sy)
    sz = rho.copy()
    sz[:spin_states,:spin_states] = jmat(1,'z').full()
    sz[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    sz = Qobj(sz)
    s_plus = rho.copy()
    s_plus[:spin_states,:spin_states] = jmat(1,'+').full()
    s_plus[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    s_plus = Qobj(s_plus)
    s_minus = rho.copy()
    s_minus[:spin_states,:spin_states] = jmat(1,'-').full()
    s_minus[spin_states:,spin_states:] = qeye(n_states-spin_states).full()
    s_minus = Qobj(s_minus)

    # Other Operators
    # |g> --> |e>
    excitation = rho.copy()
    excitation[3,0] = np.sqrt(gamma_p)    # why this term is related to the specific transition? 
    excitation[4,1] = np.sqrt(gamma_p)
    excitation[5,2] = np.sqrt(gamma_p)
    excitation = Qobj(excitation)

    # |e> --> |g>
    emission = rho.copy()
    emission[0,3] = np.sqrt(gamma_0)
    emission[1,4] = np.sqrt(gamma_0)
    emission[2,5] = np.sqrt(gamma_0)
    emission = Qobj(emission)

    # |e+1> --> |s>
    singlet_in1p = rho.copy()
    singlet_in1p[6,3] = np.sqrt(gamma_f1) 
    singlet_in1p = Qobj(singlet_in1p)

    # |e0> --> |s>
    singlet_in0 = rho.copy()
    singlet_in0[6,4] = np.sqrt(gamma_f0) 
    singlet_in0 = Qobj(singlet_in0)

    # |e-1> --> |s>
    singlet_in1m = rho.copy()
    singlet_in1m[6,5] = np.sqrt(gamma_f1) 
    singlet_in1m = Qobj(singlet_in1m)

    # |s> --> |g+1>
    singlet_out1p = rho.copy()
    singlet_out1p[0,6] = np.sqrt(gamma_s1)  
    singlet_out1p = Qobj(singlet_out1p)

    # |s> --> |g0>
    singlet_out0 = rho.copy()
    singlet_out0[1,6] = np.sqrt(gamma_s0) 
    singlet_out0 = Qobj(singlet_out0)

    # |s> --> |g-1>
    singlet_out1m = rho.copy()
    singlet_out1m[2,6] = np.sqrt(gamma_s1) 
    singlet_out1m = Qobj(singlet_out1m)

    # |e+1> --> |d>
    dark_in1p = rho.copy()
    dark_in1p[7,3] = np.sqrt(gamma_d1_in) 
    dark_in1p = Qobj(dark_in1p)

    # |e0> --> |d>
    dark_in0 = rho.copy()
    dark_in0[7,4] = np.sqrt(gamma_d0_in) 
    dark_in0 = Qobj(dark_in0)

    # |e-1> --> |d>
    dark_in1m = rho.copy()
    dark_in1m[7,5] = np.sqrt(gamma_d1_in) 
    dark_in1m = Qobj(dark_in1m)

    # |d> --> |g+1>
    dark_out1p = rho.copy()
    dark_out1p[0,7] = np.sqrt(gamma_d1_out)  
    dark_out1p = Qobj(dark_out1p)

    # |d> --> |g0>
    dark_out0 = rho.copy()
    dark_out0[1,7] = np.sqrt(gamma_d0_out) 
    dark_out0 = Qobj(dark_out0)

    # |d> --> |g-1>
    dark_out1m = rho.copy()
    dark_out1m[2,7] = np.sqrt(gamma_d1_out) 
    dark_out1m = Qobj(dark_out1m)

    # Time-independent Hamiltonian
    H0 = D*(sz**2 - 2/3) + g*sz*Bz + g*(sx*Bx + sy*By) + E*(sx**2 - sy**2)

    # Solve for eigenenergies and eigenstates
    eigenenergies, eigenstates = H0.eigenstates()

    # print(eigenenergies)
    
    eigenenergies = np.unique(eigenenergies)
    # print(eigenenergies)

    if drive == -1:
        omega = eigenenergies[1]-eigenenergies[0]-detuning_minus
        # print(f"Resonant frequency is {omega} MHz")
    else:
        omega = eigenenergies[2]-eigenenergies[0]
        # print(f"Resonant frequency is {omega} MHz")
        

    H1 = g * (sx*B1x + sy*B1y + sz*B1z) 
    H_drive = [[H1, lambda t, args: np.cos(omega * t)]]

    # Total Hamiltonian
    H = [H0] + H_drive

    # Collapse operators
    c_ops = [
            np.sqrt(gamma_rel/(2*np.pi)) * s_minus,     # T1 Relaxation (Sigma_minus)   
            np.sqrt(gamma_rel/(2*np.pi)) * s_plus,      # T1 Relaxation (sigma_plus)    
            np.sqrt(2*gamma_deph) * sz,         # Dephasing                             
            excitation,                         # Excitation
            emission,                           # Emission
        
            singlet_in1p,                       # |e+1> --> |s>
            singlet_in0,                        # |e0> --> |s>
            singlet_in1m,                       # |e-1> --> |s>
            singlet_out1p,                      # |s> --> |g+1>
            singlet_out0,                       # |s> --> |g0>
            singlet_out1m,                      # |s> --> |g-1>
        
            dark_in1p,                          # |e+1> --> |d>
            dark_in0,                           # |e0> --> |d>
            dark_in1m,                          # |e-1> --> |d>
            dark_out1p,                         # |d> --> |g+1>
            dark_out0,                          # |d> --> |g0>
            dark_out1m,                         # |d> --> |g-1>
            ]


    # Observables to record
    expect_ops = [basis(n_states, 0)*basis(n_states, 0).dag(),  # |+1_g> Population
                basis(n_states, 1)*basis(n_states, 1).dag(),    # |0_g> Population
                basis(n_states, 2)*basis(n_states, 2).dag(),    # |-1_g> Population
                basis(n_states, 3)*basis(n_states, 3).dag(),    # |+1_e> Population
                basis(n_states, 4)*basis(n_states, 4).dag(),    # |0_e> Population
                basis(n_states, 5)*basis(n_states, 5).dag(),    # |-1_e> Population
                basis(n_states, 6)*basis(n_states, 6).dag(),    # |s> Population
                basis(n_states, 7)*basis(n_states, 7).dag(),    # |d> Population
                ]

    # Solve the master equation
    result = mesolve(H, psi0, tlist, c_ops, expect_ops,
                   options={"store_final_state": True, "nsteps": 1e5, "store_states": True, "method" : 'lsoda'},
                   )
    # result = mesolve(H, psi0, tlist, c_ops, expect_ops,
    #                options={"store_final_state": True, "nsteps": 1e5, "store_states": False},
    #                )

    return result.expect, result.final_state, result.states
    # return result.expect, _, result.final_state
    
    # return eigenenergies  

In [7]:
# def delay(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
#                gamma_rel=0.0001, gamma_deph=0.001, power_laser=0, B1=0, duration=5, drive=-1, t_steps=1001, gamma_d_in=0,gamma_d_out=0):
    
#     parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out)
    
#     # Time array
#     tlist = np.linspace(0, duration, t_steps)  # Time in us
    
#     populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)
    
#     return populations, psi_list, tlist   

# ########################################################################################################

def initialize(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
               gamma_rel=0.0001, gamma_deph=0.001, power_laser=100, B1=0, duration=5, drive=-1, t_steps=1001, 
               gamma_d_in=0,gamma_d_out=0,exc_rate=0.1,rel_rate=10,s0_in=12,s1_in=80,s0_out=3.3,s1_out=2.4):
    
    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out,
                               exc_rate=exc_rate,rel_rate=rel_rate,s0_in=s0_in,s1_in=s1_in,s0_out=s0_out,s1_out=s1_out)
    
    # Time array
    tlist = np.linspace(0, duration, t_steps)  # Time in us
    
    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)
    
    return populations, psi_list, tlist   

########################################################################################################

def rabi(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
               gamma_rel=0.0001, gamma_deph=0.001, power_laser=0, B1=0.00164, duration=5, 
         drive=-1, theta=np.pi/2, phi=0, t_steps=101,detuning_minus=0.0, gamma_d_in=0,gamma_d_out=0,exc_rate=0.1,rel_rate=10,s0_in=12,s1_in=80,s0_out=3.3,s1_out=2.4):
    # print('value of B1 in function rabi: ',B1)
    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1, theta, phi,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out,
                              exc_rate=exc_rate,rel_rate=rel_rate,s0_in=s0_in,s1_in=s1_in,s0_out=s0_out,s1_out=s1_out)
    
    # Time array
    tlist = np.linspace(0, duration, t_steps)  # Time in us
    
    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive,detuning_minus)
    
    return populations, psi_list, tlist  

########################################################################################################

def evolve(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
           exp_t=None,
           gamma_rel=0.0001, gamma_deph=0.001, power_laser=0, B1=0,
           duration=100, drive=-1, t_steps=101, spacing_type='linspace',
           gamma_d_in=0, gamma_d_out=0, t_start=0.02,
           exc_rate=0.1, rel_rate=10, s0_in=12, s1_in=80, s0_out=3.3, s1_out=2.4):

    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser,
                               gamma_d_in=gamma_d_in, gamma_d_out=gamma_d_out,
                               exc_rate=exc_rate, rel_rate=rel_rate,
                               s0_in=s0_in, s1_in=s1_in, s0_out=s0_out, s1_out=s1_out)

    if exp_t is not None:
        tlist = exp_t
    else:
        duration = duration
        if spacing_type == 'geomspace':
            tlist = np.geomspace(t_start, duration, t_steps)
        else:
            tlist = np.linspace(0, duration, t_steps)

    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)

    return populations, psi_list, tlist
 

########################################################################################################

def read(psi0=(1/3)*ket2dm(basis(8, 0)) + (1/3)*ket2dm(basis(8, 1)) + (1/3)*ket2dm(basis(8, 2)),
               gamma_rel=0.0001, gamma_deph=0.001, power_laser=100, B1=0, duration=5, drive=-1, t_steps=1001, 
         gamma_d_in=0,gamma_d_out=0,exc_rate=0.1,rel_rate=10,s0_in=12,s1_in=80,s0_out=3.3,s1_out=2.4):
    
    parameters = nv_parameters(gamma_rel, gamma_deph, power_laser, B1,gamma_d_in=gamma_d_in,gamma_d_out=gamma_d_out,
                              exc_rate=exc_rate,rel_rate=rel_rate,s0_in=s0_in,s1_in=s1_in,s0_out=s0_out,s1_out=s1_out)
    psi0 = Qobj(qdiags(psi0.diag(), offsets=0), dims=psi0.dims)        # removing the off-diagonal terms as they are not relevant                
    # Time array
    tlist = np.linspace(0, duration, t_steps)  # Time in us
    
    populations, _, psi_list = NV_spinNcharge(parameters, tlist, psi0, drive)
    
    return populations, psi_list, tlist  

########################################################################################################

def count(populations,tlist,duration=0.5, gamma_0=63, collection=0.01):
    index = np.argwhere(tlist<=duration)[-1][0]
    # print(index)
    n = np.mean(np.array(np.real(populations[3:6])[:,:index+1]))
    photons = n*duration*gamma_0*collection
    return photons

#### Actual $T_1$

In [8]:
# # power_array = np.array([5,15,24])
# # init_array = np.array([15,50,100])
# power_array = np.array([8])
# init_array = np.array([25])

# gamma_d_in = np.linspace(8,500,20)
# gamma_d_out = np.linspace(0.3,30,20)
# # gamma_d_in = np.array([0])
# # gamma_d_out = np.array([0])

# dark_steps = 51

In [9]:
# # Parameters
# gamma_rel = 0.02      # keeping T1 lifetime = 50 us
# gamma_deph = 0.0     
# read_window = 0.3
# dark_time = 100

# signal_array = np.zeros((power_array.shape[0],init_array.shape[0], gamma_d_in.shape[0],gamma_d_out.shape[0],dark_steps))
# t_array = np.zeros((power_array.shape[0],init_array.shape[0], gamma_d_in.shape[0],gamma_d_out.shape[0],dark_steps))


# for p, laser_power in tqdm(enumerate(power_array), total=len(power_array), position=0, leave=True, desc="power_array"):
#     for t, t_init in tqdm(enumerate(init_array), total=len(init_array), position=0, leave=True, desc="init_array"):
#         for d1, d_in in tqdm(enumerate(gamma_d_in), total=len(gamma_d_in), position=0, leave=True, desc="gamma_d_in"):
#             for d2, d_out in tqdm(enumerate(gamma_d_out), total=len(gamma_d_out), position=0, leave=True, desc="gamma_d_out"):

                
#                 _,psi1,_=initialize(psi0,power_laser=laser_power,duration=t_init,t_steps=51,
#                                     gamma_rel=gamma_rel,gamma_deph=gamma_deph, gamma_d_in=d_in,gamma_d_out=d_out)
#                 populations1,psi2,t_t1=evolve(psi1[-1], duration=dark_time,t_steps=dark_steps,
#                                               spacing_type = 'geomspace', gamma_rel=gamma_rel,gamma_deph=gamma_deph, gamma_d_in=d_in,gamma_d_out=d_out)
#                 t_array[p,t,d1,d2,:] = t_t1
                
#                 for i in range(len(psi2)):
#                     populations3,psi3,t_read=read(psi2[i],power_laser=laser_power,duration=read_window,t_steps=51,
#                                                   gamma_rel=gamma_rel,gamma_deph=gamma_deph, gamma_d_in=d_in,gamma_d_out=d_out)
#                     photons = count(populations3, t_read, duration=read_window) 
#                     signal_array[p,t,d1,d2,i] = photons

In [10]:
exp_filename = r"D:\Atanu\Atanu_Github\T1_measurements\codes\data\2025\May_12\arranging_datas_with_power\5uW\l_15.0us\[17_22_10]_1.0us.npz"
exp_data = np.load(exp_filename)
time_axis, signal_photon, _, _ = data_to_time_signal(exp_data, 2000, first='signal')

time_axis = time_axis[1:]
signal_photon = signal_photon[1:]

signal_photon.shape,time_axis.shape

((49,), (49,))

In [11]:
t_array = time_axis.copy()
t_array = t_array/1e3       # as simulation time array needs to be in microsecond
dark_steps = len(t_array)  

In [12]:
# for t_array same to the experimental time axis

from joblib import Parallel, delayed

# === Parameters ===
power_array = np.array([5])
init_array = np.array([15])
gamma_rel = np.array([0.02])
gamma_d_in = np.linspace(0, 40, 1)
gamma_d_out = np.linspace(0.0299, 0.035, 1)
exc_rate = np.linspace(0.05, 0.9, 4)
rel_rate = np.linspace(5, 60, 4)
s0_in = np.linspace(10, 15, 4)
s1_in = np.linspace(18, 100, 4)
s0_out = np.linspace(3.0, 6, 4)
s1_out = np.linspace(1.0, 2.8, 4)
gamma_deph = 0.0
read_window = 1
dark_time = 100
# dark_steps = 51  # Now we generate t_array inside compute_signal

# === Initial State ===
power = power_array[0]
dark_ratio = 0.5 / power
prob_dark = dark_ratio * power
prob_nv = 1 - prob_dark
psi0 = ((prob_nv / 3) * ket2dm(basis(8, 0)) +
        (prob_nv / 3) * ket2dm(basis(8, 1)) +
        (prob_nv / 3) * ket2dm(basis(8, 2)) +
        prob_dark * ket2dm(basis(8, 7)))

# === Output Array ===
signal_array = np.zeros((
    power_array.shape[0],
    init_array.shape[0],
    gamma_d_in.shape[0],
    gamma_d_out.shape[0],
    exc_rate.shape[0],
    rel_rate.shape[0],
    s0_in.shape[0],
    s1_in.shape[0],
    s0_out.shape[0],
    s1_out.shape[0],
    dark_steps
))

# === Compute Signal ===
def compute_signal(args):
    try:
        params, psi0, power_array, init_array, \
        gamma_d_in, gamma_d_out, exc_rate, rel_rate, s0_in, s1_in, s0_out, s1_out, \
        gamma_rel, gamma_deph, read_window, dark_time, t_array = args

        p, t, d1, d2, e, r, si0, si1, so0, so1 = params

        laser_power = power_array[p]
        t_init = init_array[t]
        d_in = gamma_d_in[d1]
        d_out = gamma_d_out[d2]
        exc = exc_rate[e]
        rel = rel_rate[r]
        s0i = s0_in[si0]
        s1i = s1_in[si1]
        s0o = s0_out[so0]
        s1o = s1_out[so1]

        _, psi1, _ = initialize(
            psi0, power_laser=laser_power, duration=t_init, t_steps=51,
            gamma_rel=gamma_rel, gamma_deph=gamma_deph,
            gamma_d_in=d_in, gamma_d_out=d_out,
            exc_rate=exc, rel_rate=rel,
            s0_in=s0i, s1_in=s1i, s0_out=s0o, s1_out=s1o
        )

        _, psi2, _ = evolve(
            psi1[-1], exp_t=t_array,
            gamma_rel=gamma_rel, gamma_deph=gamma_deph,
            gamma_d_in=d_in, gamma_d_out=d_out,
            exc_rate=exc, rel_rate=rel,
            s0_in=s0i, s1_in=s1i, s0_out=s0o, s1_out=s1o
        )

        signal_local = np.zeros(len(t_array))
        for i, psi in enumerate(psi2):
            populations3, _, t_read = read(
                psi, power_laser=laser_power, duration=read_window, t_steps=51,
                gamma_rel=gamma_rel, gamma_deph=gamma_deph,
                gamma_d_in=d_in, gamma_d_out=d_out,
                exc_rate=exc, rel_rate=rel,
                s0_in=s0i, s1_in=s1i, s0_out=s0o, s1_out=s1o
            )
            photons = count(populations3, t_read, duration=read_window)
            signal_local[i] = photons

        return (p, t, d1, d2, e, r, si0, si1, so0, so1, signal_local)

    except Exception as e:
        import traceback
        print(f"Error in params {params}: {e}")
        traceback.print_exc()
        raise

# === Param List ===
params_list = [
    (p, t, d1, d2, e, r, si0, si1, so0, so1)
    for p in range(len(power_array))
    for t in range(len(init_array))
    for d1 in range(len(gamma_d_in))
    for d2 in range(len(gamma_d_out))
    for e in range(len(exc_rate))
    for r in range(len(rel_rate))
    for si0 in range(len(s0_in))
    for si1 in range(len(s1_in))
    for so0 in range(len(s0_out))
    for so1 in range(len(s1_out))
]

args_list = [
    (params, psi0, power_array, init_array, gamma_d_in, gamma_d_out,
     exc_rate, rel_rate, s0_in, s1_in, s0_out, s1_out,
     gamma_rel, gamma_deph, read_window, dark_time, t_array)
    for params in params_list
]

# === Run Parallel Simulation ===
results = Parallel(n_jobs=-1)(
    delayed(compute_signal)(arg) for arg in tqdm(args_list)
)

# === Store Results ===
for p, t, d1, d2, e, r, si0, si1, so0, so1, signal_local in results:
    signal_array[p, t, d1, d2, e, r, si0, si1, so0, so1, :] = signal_local

print("Completed processing.")

  2%|█▎                                                                            | 72/4096 [01:03<1:01:38,  1.09it/s]

KeyboardInterrupt: 

In [ ]:
## for user defined time array 


# from joblib import Parallel, delayed

# # === Parameters ===
# power_array = np.array([5])
# init_array = np.array([15])
# gamma_rel = np.array([0.02])
# gamma_d_in = np.linspace(0, 40, 1)
# gamma_d_out = np.linspace(0.0299, 0.035, 1)
# exc_rate = np.linspace(0.7, 0.05, 1)
# rel_rate = np.linspace(10, 60, 1)
# s0_in = np.linspace(12, 15, 1)
# s1_in = np.linspace(50, 100, 1)
# s0_out = np.linspace(3.3, 6, 1)
# s1_out = np.linspace(2.4, 3.0, 1)
# gamma_deph = 0.0
# read_window = 1
# dark_time = 100
# dark_steps = 51  # Now we generate t_array inside compute_signal

# # === Initial State ===
# power = power_array[0]
# dark_ratio = 0.5 / power
# prob_dark = dark_ratio * power
# prob_nv = 1 - prob_dark
# psi0 = ((prob_nv / 3) * ket2dm(basis(8, 0)) +
#         (prob_nv / 3) * ket2dm(basis(8, 1)) +
#         (prob_nv / 3) * ket2dm(basis(8, 2)) +
#         prob_dark * ket2dm(basis(8, 7)))

# # === Output Arrays ===
# signal_array = np.zeros((
#     len(power_array),
#     len(init_array),
#     len(gamma_d_in),
#     len(gamma_d_out),
#     len(exc_rate),
#     len(rel_rate),
#     len(s0_in),
#     len(s1_in),
#     len(s0_out),
#     len(s1_out),
#     dark_steps
# ))

# t_array = np.zeros_like(signal_array)  # Same shape to store per-parameter time arrays


# # === Compute Signal ===
# def compute_signal(args):
#     try:
#         params, psi0, power_array, init_array, \
#         gamma_d_in, gamma_d_out, exc_rate, rel_rate, s0_in, s1_in, s0_out, s1_out, \
#         gamma_rel, gamma_deph, read_window, dark_time, dark_steps = args

#         p, t, d1, d2, e, r, si0, si1, so0, so1 = params

#         laser_power = power_array[p]
#         t_init = init_array[t]
#         d_in = gamma_d_in[d1]
#         d_out = gamma_d_out[d2]
#         exc = exc_rate[e]
#         rel = rel_rate[r]
#         s0i = s0_in[si0]
#         s1i = s1_in[si1]
#         s0o = s0_out[so0]
#         s1o = s1_out[so1]

#         # === Initialization ===
#         _, psi1, _ = initialize(
#             psi0, power_laser=laser_power, duration=t_init, t_steps=51,
#             gamma_rel=gamma_rel, gamma_deph=gamma_deph,
#             gamma_d_in=d_in, gamma_d_out=d_out,
#             exc_rate=exc, rel_rate=rel,
#             s0_in=s0i, s1_in=s1i, s0_out=s0o, s1_out=s1o
#         )

#         # === Dark Evolution ===
#         populations1, psi2, t_t1 = evolve(
#             psi1[-1], duration=dark_time, t_steps=dark_steps,
#             spacing_type='geomspace', t_start=0.02,
#             gamma_rel=gamma_rel, gamma_deph=gamma_deph,
#             gamma_d_in=d_in, gamma_d_out=d_out,
#             exc_rate=exc, rel_rate=rel,
#             s0_in=s0i, s1_in=s1i, s0_out=s0o, s1_out=s1o
#         )

#         signal_local = np.zeros(dark_steps)
#         for i, psi in enumerate(psi2):
#             populations3, _, t_read = read(
#                 psi, power_laser=laser_power, duration=read_window, t_steps=51,
#                 gamma_rel=gamma_rel, gamma_deph=gamma_deph,
#                 gamma_d_in=d_in, gamma_d_out=d_out,
#                 exc_rate=exc, rel_rate=rel,
#                 s0_in=s0i, s1_in=s1i, s0_out=s0o, s1_out=s1o
#             )
#             photons = count(populations3, t_read, duration=read_window)
#             signal_local[i] = photons

#         return (p, t, d1, d2, e, r, si0, si1, so0, so1, t_t1, signal_local)

#     except Exception as e:
#         import traceback
#         print(f"Error in params {params}: {e}")
#         traceback.print_exc()
#         raise

# # === Param List ===
# params_list = [
#     (p, t, d1, d2, e, r, si0, si1, so0, so1)
#     for p in range(len(power_array))
#     for t in range(len(init_array))
#     for d1 in range(len(gamma_d_in))
#     for d2 in range(len(gamma_d_out))
#     for e in range(len(exc_rate))
#     for r in range(len(rel_rate))
#     for si0 in range(len(s0_in))
#     for si1 in range(len(s1_in))
#     for so0 in range(len(s0_out))
#     for so1 in range(len(s1_out))
# ]

# args_list = [
#     (params, psi0, power_array, init_array, gamma_d_in, gamma_d_out,
#      exc_rate, rel_rate, s0_in, s1_in, s0_out, s1_out,
#      gamma_rel, gamma_deph, read_window, dark_time, dark_steps)
#     for params in params_list
# ]

# # === Run Simulation ===
# results = Parallel(n_jobs=-1)(
#     delayed(compute_signal)(arg) for arg in tqdm(args_list)
# )

# # === Store Results ===
# for p, t, d1, d2, e, r, si0, si1, so0, so1, t_t1, signal_local in results:
#     signal_array[p, t, d1, d2, e, r, si0, si1, so0, so1, :] = signal_local
#     t_array[p, t, d1, d2, e, r, si0, si1, so0, so1, :] = t_t1

# print("Completed processing.")


In [ ]:
# === Save Results ===
folder_path = 'data'
os.makedirs(folder_path, exist_ok=True)
filename = os.path.join(folder_path, f'5uW_15us_1us_3points.npz')
np.savez_compressed(filename, signal_array=signal_array, t_array=t_array)

loaded = np.load(filename)
signal_array = loaded['signal_array']
t_array = loaded['t_array']

In [ ]:
signal_array.shape, t_array.shape

In [ ]:
import plotly.graph_objs as go

# --- Prepare simulation data ---
sim_signal = signal_array[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, :]
t_signal = t_array*1e3   # to convert to microsecond

# --- Normalize both signals (optional) ---
signal_photon_norm = signal_photon / np.max(signal_photon)
sim_signal_norm = sim_signal / np.max(sim_signal)

# --- Create figure ---
fig = go.Figure()

# --- Plot experimental data ---
fig.add_trace(go.Scatter(
    x=time_axis,
    # y=signal_photon_norm,
    y=signal_photon,
    
    mode='markers',
    name='Experiment',
    marker=dict(color='royalblue', size=6),
    line=dict(color='royalblue', width=2),
))

# --- Plot simulated data ---
fig.add_trace(go.Scatter(
    x=t_signal,
    # y=sim_signal_norm,
    y=sim_signal,
    
    mode='markers',
    name='Simulation',
    line=dict(color='crimson', width=3),
))

# --- Axis labels and layout ---
fig.update_layout(
    template=fig_template,
    width=800,
    height=600,
    xaxis=dict(
        title='Time (ns)',
        range=[20, 1000],  # Set x-axis range
        type='linear'
    ),
    yaxis=dict(
        title='Normalized Signal',
    )
)

# --- Show plot ---
fig.show()

In [ ]:
import numpy as np
from scipy.optimize import differential_evolution

# --- Helper: Find closest index in array ---
def find_index(arr, val):
    return int(np.argmin(np.abs(arr - val)))

# --- Cost function with overestimation penalty ---
def mae_cost_log_params_penalty(x, signal_array, signal_photon, t_array, time_axis):
    scale = np.exp(x[0])
    exc = np.exp(x[1])
    rel = np.exp(x[2])
    s0i = x[3]
    s1i = x[4]
    s0o = x[5]
    s1o = x[6]
    
    e = find_index(exc_rate, exc)
    r = find_index(rel_rate, rel)
    si0_idx = find_index(s0_in, s0i)
    si1_idx = find_index(s1_in, s1i)
    so0_idx = find_index(s0_out, s0o)
    so1_idx = find_index(s1_out, s1o)
    
    sim_signal = signal_array[0, 0, 0, 0, e, r, si0_idx, si1_idx, so0_idx, so1_idx, :]
    sim_signal_scaled = sim_signal * scale
    
    sim_time_ns = t_array * 1e3
    sim_interp = np.interp(time_axis, sim_time_ns, sim_signal_scaled)
    
    mask = (time_axis >= 20) & (time_axis <= 1000)
    exp = signal_photon[mask]
    sim = sim_interp[mask]
    
    mae = np.mean(np.abs(exp - sim))
    
    # Penalize overestimation (simulation > experiment)
    over_penalty = np.mean(np.maximum(sim - exp, 0)) * 10  # penalty weight
    
    return mae + over_penalty

# --- Initial guess ---
x0 = [
    np.log(400),  # Initial scale factor guess (since experiment ~400x sim)
    np.log(0.7),  # exc_rate
    np.log(10),   # rel_rate
    12,           # s0_in
    50,           # s1_in
    3.3,          # s0_out
    2.4           # s1_out
]

# --- Bounds ---
bounds = [
    (np.log(1), np.log(200)),               # tighter scale factor bounds
    (np.log(min(exc_rate)), np.log(max(exc_rate))),
    (np.log(min(rel_rate)), np.log(max(rel_rate))),
    (min(s0_in), max(s0_in)),
    (min(s1_in), max(s1_in)),
    (min(s0_out), max(s0_out)),
    (min(s1_out), max(s1_out)),
]

# --- Run Differential Evolution optimization (quiet mode) ---
result = differential_evolution(
    mae_cost_log_params_penalty,
    bounds=bounds,
    args=(signal_array, signal_photon, t_array, time_axis),
    maxiter=1000,
    polish=True,
    disp=False,       # no verbose output
    seed=42
)

# --- Convert optimized parameters back from log ---
optimized_params = result.x.copy()
optimized_params[0] = np.exp(optimized_params[0])  # scale
optimized_params[1] = np.exp(optimized_params[1])  # exc_rate
optimized_params[2] = np.exp(optimized_params[2])  # rel_rate
# The remaining params are linear, no change needed

print("Optimized parameters (normal scale):")
print(f"Scale factor: {optimized_params[0]:.4f}")
print(f"Excitation rate (exc_rate): {optimized_params[1]:.4f}")
print(f"Relaxation rate (rel_rate): {optimized_params[2]:.4f}")
print(f"s0_in: {optimized_params[3]:.4f}")
print(f"s1_in: {optimized_params[4]:.4f}")
print(f"s0_out: {optimized_params[5]:.4f}")
print(f"s1_out: {optimized_params[6]:.4f}")

print("Minimum MAE:", result.fun)

In [ ]:
import plotly.graph_objs as go

# --- Extract optimized parameters ---
scale_opt = np.exp(result.x[0])
exc_opt = np.exp(result.x[1])
rel_opt = np.exp(result.x[2])
s0i_opt = result.x[3]
s1i_opt = result.x[4]
s0o_opt = result.x[5]
s1o_opt = result.x[6]

# --- Find closest indices ---
e_opt = find_index(exc_rate, exc_opt)
r_opt = find_index(rel_rate, rel_opt)
si0_opt = find_index(s0_in, s0i_opt)
si1_opt = find_index(s1_in, s1i_opt)
so0_opt = find_index(s0_out, s0o_opt)
so1_opt = find_index(s1_out, s1o_opt)

# --- Get optimized simulation signal ---
sim_signal_opt = signal_array[0, 0, 0, 0, e_opt, r_opt, si0_opt, si1_opt, so0_opt, so1_opt, :]
sim_signal_scaled = sim_signal_opt * scale_opt

# --- Also get initial guess signals for comparison ---
x0 = [
    np.log(400),
    np.log(0.7),
    np.log(10),
    12,
    50,
    3.3,
    2.4
]

scale_init = np.exp(x0[0])
exc_init = np.exp(x0[1])
rel_init = np.exp(x0[2])
s0i_init = x0[3]
s1i_init = x0[4]
s0o_init = x0[5]
s1o_init = x0[6]

e_init = find_index(exc_rate, exc_init)
r_init = find_index(rel_rate, rel_init)
si0_init = find_index(s0_in, s0i_init)
si1_init = find_index(s1_in, s1i_init)
so0_init = find_index(s0_out, s0o_init)
so1_init = find_index(s1_out, s1o_init)

sim_signal_init = signal_array[0, 0, 0, 0, e_init, r_init, si0_init, si1_init, so0_init, so1_init, :]
sim_signal_init_scaled = sim_signal_init * scale_init

# --- Interpolate to experimental time axis (ns) ---
sim_time_ns = t_array * 1e3
sim_interp_opt = np.interp(time_axis, sim_time_ns, sim_signal_scaled)
sim_interp_init = np.interp(time_axis, sim_time_ns, sim_signal_init_scaled)

# --- Create Plot ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=time_axis, y=signal_photon,
    mode='markers+lines',
    name='Experimental',
    line=dict(color='black', dash='dot'),
    marker=dict(symbol='circle', size=6)
))

fig.add_trace(go.Scatter(
    x=time_axis, y=sim_interp_init,
    mode='lines',
    name='Initial Simulation',
    line=dict(color='blue', dash='dash')
))

fig.add_trace(go.Scatter(
    x=time_axis, y=sim_interp_opt,
    mode='lines',
    name='Optimized Fit',
    line=dict(color='red')
))

fig.update_layout(
    template=fig_template,
    height=700, width=800,
    xaxis=dict(title="Time (ns)"),
    yaxis=dict(title="Photon Counts (arb. units)"),
    title="Experimental vs Simulation Fit with Scale Penalty"
)

fig.update_yaxes(automargin=True)

fig.show()